In [0]:
# Databricks notebook source
# Controllo generale del progetto Meteo ed Energia.

from pyspark.sql import functions as F


# Imposto catalogo e tabelle.
catalogo = "progetto_meteo"

tabella_citta = f"{catalogo}.metadati.citta_monitorate"
tabella_zone_entsoe = f"{catalogo}.metadati.zone_entsoe"

tabella_meteo_storico = f"{catalogo}.bronze.meteo_storico"
tabella_meteo_streaming = f"{catalogo}.bronze.meteo_streaming"
tabella_dati_entsoe = f"{catalogo}.bronze.dati_entsoe"

tabella_dati_aggiornati = f"{catalogo}.silver.dati_aggiornati"
tabella_energy_block = f"{catalogo}.silver.energy_block"

tabella_dati_aggregati = f"{catalogo}.gold.dati_aggregati"
tabella_dati_studio = f"{catalogo}.gold.dati_studio"

spark.sql(f"USE CATALOG {catalogo}")


# Creo una funzione semplice per contare le righe.
def conta_righe(tabella):

    try:
        return spark.table(tabella).count()
    except:
        return None


# Controllo quante righe ho in tutte le tabelle principali.
tabelle = [
    tabella_citta,
    tabella_zone_entsoe,
    tabella_meteo_storico,
    tabella_meteo_streaming,
    tabella_dati_entsoe,
    tabella_dati_aggiornati,
    tabella_energy_block,
    tabella_dati_aggregati,
    tabella_dati_studio
]

righe_tabelle = []

for tabella in tabelle:
    righe_tabelle.append({
        "Tabella": tabella,
        "Righe": conta_righe(tabella)
    })

df_controllo_tabelle = spark.createDataFrame(righe_tabelle)

display(df_controllo_tabelle)

print("Controllo righe completato.")


# Controllo le città monitorate per area.
display(
    spark.table(tabella_citta)
    .groupBy("area")
    .count()
    .orderBy("area")
)

print("Città monitorate:", spark.table(tabella_citta).count())


# Controllo le zone ENTSO-E per area.
display(
    spark.table(tabella_zone_entsoe)
    .groupBy("Area")
    .count()
    .orderBy("Area")
)

print("Zone ENTSO-E:", spark.table(tabella_zone_entsoe).count())


# Controllo il periodo coperto dal meteo storico.
if conta_righe(tabella_meteo_storico) > 0:

    display(
        spark.table(tabella_meteo_storico)
        .agg(
            F.min("Data").alias("Data_Min"),
            F.max("Data").alias("Data_Max"),
            F.count("*").alias("Righe")
        )
    )

    display(
        spark.table(tabella_meteo_storico)
        .groupBy(F.year("Data").alias("Anno"))
        .count()
        .orderBy("Anno")
    )

else:
    print("meteo_storico è vuota.")


# Controllo il periodo coperto dalla Silver meteo.
if conta_righe(tabella_dati_aggiornati) > 0:

    display(
        spark.table(tabella_dati_aggiornati)
        .agg(
            F.min("Data").alias("Data_Min"),
            F.max("Data").alias("Data_Max"),
            F.count("*").alias("Righe")
        )
    )

    display(
        spark.table(tabella_dati_aggiornati)
        .groupBy(F.year("Data").alias("Anno"))
        .count()
        .orderBy("Anno")
    )

else:
    print("dati_aggiornati è vuota.")


# Controllo il periodo coperto da ENTSO-E Bronze.
if conta_righe(tabella_dati_entsoe) > 0:

    display(
        spark.table(tabella_dati_entsoe)
        .agg(
            F.min("Start").alias("Start_Min"),
            F.max("Start").alias("Start_Max"),
            F.count("*").alias("Righe")
        )
    )

    display(
        spark.table(tabella_dati_entsoe)
        .groupBy(F.year("Data").alias("Anno"))
        .count()
        .orderBy("Anno")
    )

else:
    print("dati_entsoe è vuota.")


# Controllo il periodo coperto da Energy Block.
# Qui Data è stringa dd/MM/yyyy, quindi la converto prima di fare min/max.
if conta_righe(tabella_energy_block) > 0:

    df_energy_block_controllo = (
        spark.table(tabella_energy_block)
        .withColumn("Data_Corretta", F.to_date("Data", "dd/MM/yyyy"))
    )

    display(
        df_energy_block_controllo
        .agg(
            F.min("Data_Corretta").alias("Data_Min"),
            F.max("Data_Corretta").alias("Data_Max"),
            F.count("*").alias("Righe")
        )
    )

    display(
        df_energy_block_controllo
        .groupBy(F.year("Data_Corretta").alias("Anno"))
        .count()
        .orderBy("Anno")
    )

else:
    print("energy_block è vuota.")


# Controllo il periodo coperto da dati_aggregati.
# Anche qui Data è stringa dd/MM/yyyy.
if conta_righe(tabella_dati_aggregati) > 0:

    df_dati_aggregati_controllo = (
        spark.table(tabella_dati_aggregati)
        .withColumn("Data_Corretta", F.to_date("Data", "dd/MM/yyyy"))
    )

    display(
        df_dati_aggregati_controllo
        .agg(
            F.min("Data_Corretta").alias("Data_Min"),
            F.max("Data_Corretta").alias("Data_Max"),
            F.count("*").alias("Righe")
        )
    )

    display(
        df_dati_aggregati_controllo
        .groupBy(F.year("Data_Corretta").alias("Anno"))
        .count()
        .orderBy("Anno")
    )

else:
    print("dati_aggregati è vuota.")


# Controllo il periodo coperto da dati_studio.
# Anche qui Data è stringa dd/MM/yyyy.
if conta_righe(tabella_dati_studio) > 0:

    df_dati_studio_controllo = (
        spark.table(tabella_dati_studio)
        .withColumn("Data_Corretta", F.to_date("Data", "dd/MM/yyyy"))
    )

    display(
        df_dati_studio_controllo
        .agg(
            F.min("Data_Corretta").alias("Data_Min"),
            F.max("Data_Corretta").alias("Data_Max"),
            F.count("*").alias("Righe")
        )
    )

    display(
        df_dati_studio_controllo
        .groupBy(F.year("Data_Corretta").alias("Anno"))
        .count()
        .orderBy("Anno")
    )

else:
    print("dati_studio è vuota.")


# Controllo quante righe finali ho per area.
if conta_righe(tabella_dati_studio) > 0:

    display(
        spark.table(tabella_dati_studio)
        .groupBy("Area")
        .count()
        .orderBy("Area")
    )

else:
    print("Impossibile controllare le aree: dati_studio è vuota.")


# Controllo i null nella tabella finale.
if conta_righe(tabella_dati_studio) > 0:

    df_studio = spark.table(tabella_dati_studio)

    display(
        df_studio.select([
            F.count(F.when(F.col(colonna).isNull(), colonna)).alias(colonna)
            for colonna in df_studio.columns
        ])
    )

else:
    print("Impossibile controllare i null: dati_studio è vuota.")


# Controllo eventuali duplicati nella Gold finale.
if conta_righe(tabella_dati_studio) > 0:

    df_duplicati = (
        spark.table(tabella_dati_studio)
        .groupBy("Area", "Data", "Ora")
        .count()
        .filter(F.col("count") > 1)
    )

    duplicati = df_duplicati.count()

    if duplicati == 0:
        print("Nessun duplicato trovato in dati_studio su Area, Data, Ora.")
    else:
        print(f"Duplicati trovati in dati_studio: {duplicati}")
        display(
            df_duplicati
            .withColumn("Data_Corretta", F.to_date("Data", "dd/MM/yyyy"))
            .orderBy("Area", "Data_Corretta", "Ora")
            .drop("Data_Corretta")
        )

else:
    print("Impossibile controllare duplicati: dati_studio è vuota.")


# Controllo le prime righe finali.
if conta_righe(tabella_dati_studio) > 0:

    display(
        spark.table(tabella_dati_studio)
        .withColumn("Data_Corretta", F.to_date("Data", "dd/MM/yyyy"))
        .select(
            "Area",
            "Data",
            "Data_Corretta",
            "Ora",
            "Temperatura",
            "Umidita",
            "Velocita_Vento",
            "Precipitazioni",
            "Solare",
            "Idrico",
            "Eolico"
        )
        .orderBy("Area", "Data_Corretta", "Ora")
        .drop("Data_Corretta")
        .limit(100)
    )

else:
    print("Nessuna riga da mostrare in dati_studio.")


# Controllo le ultime righe finali.
if conta_righe(tabella_dati_studio) > 0:

    display(
        spark.table(tabella_dati_studio)
        .withColumn("Data_Corretta", F.to_date("Data", "dd/MM/yyyy"))
        .select(
            "Area",
            "Data",
            "Data_Corretta",
            "Ora",
            "Temperatura",
            "Umidita",
            "Velocita_Vento",
            "Precipitazioni",
            "Solare",
            "Idrico",
            "Eolico"
        )
        .orderBy(F.col("Data_Corretta").desc(), "Area", F.col("Ora").desc())
        .drop("Data_Corretta")
        .limit(100)
    )

else:
    print("Nessuna riga finale recente da mostrare in dati_studio.")


print("Tester completato.")